#Ali Hassan
#1011
#lab4

In [4]:
# ============================================================
# Lab 04 - Activity 1: KNN for Credit Card Fraud Detection
# Student Name : Ali Hassan
# Roll Number  : 23-NTU-CS-1011
# Dataset      : creditcard.csv (Kaggle Credit Card Fraud Dataset)
# Platform     : Google Colab
# Note         : Upload creditcard.csv before running the cells
# ============================================================


# Cell 1: Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from imblearn.over_sampling import SMOTE


# Cell 2: Read Dataset and Separate Features and Target
df = pd.read_csv('creditcardfraud-metadata.json')

print("Dataset dimensions:", df.shape)
print("\nNumber of samples in each class:\n", df['Class'].value_counts())
print("Percentage of fraud transactions: {:.3f}%".format(
    df['Class'].mean() * 100
))

# Input variables and output label
X = df.drop('Class', axis=1)
y = df['Class']  # 0 represents normal transactions; 1 represents fraud


# Cell 3: Divide Data into Training and Testing Sets
# Stratification preserves the original class proportions.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)


# Cell 4: Normalize Features
# Fit the scaler on training data to prevent data leakage.
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# Cell 5: Balance Training Data with SMOTE
# Synthetic samples are generated only for the training set.
smote = SMOTE(random_state=42)

X_train_sm, y_train_sm = smote.fit_resample(
    X_train_scaled, y_train
)

y_train_sm = np.asarray(y_train_sm)

print("Class distribution before resampling:", np.bincount(y_train))
print("Class distribution after resampling: ", np.bincount(y_train_sm))


# Cell 6: Evaluate Basic KNN Without SMOTE
# Use five neighbors as the initial model setting.
knn_base = KNeighborsClassifier(
    n_neighbors=5,
    n_jobs=-1
)

knn_base.fit(X_train_scaled, y_train)
y_pred_base = knn_base.predict(X_test_scaled)

acc_base = accuracy_score(y_test, y_pred_base)
f1_base = f1_score(y_test, y_pred_base)

print("Basic KNN - Accuracy:", acc_base)
print("Basic KNN - F1-score:", f1_base)


# Cell 7: Apply KNN to the SMOTE-Balanced Dataset
knn5 = KNeighborsClassifier(
    n_neighbors=5,
    n_jobs=-1
)

knn5.fit(X_train_sm, y_train_sm)
y_pred5 = knn5.predict(X_test_scaled)

acc5 = accuracy_score(y_test, y_pred5)
f1_5 = f1_score(y_test, y_pred5)

print("SMOTE with KNN (k=5) - Accuracy:", acc5)
print("SMOTE with KNN (k=5) - F1-score:", f1_5)


# Cell 8: Search for the Most Suitable k Value
# A subset is used to reduce the computation time of grid search.
rng = np.random.RandomState(42)
idx = rng.choice(len(X_train_sm), 20000, replace=False)

X_tune = X_train_sm[idx]
y_tune = y_train_sm[idx]

param_grid = {
    'n_neighbors': [1, 3, 5, 7, 9, 11]
}

grid = GridSearchCV(
    KNeighborsClassifier(n_jobs=-1),
    param_grid,
    cv=3,
    scoring='f1'
)

grid.fit(X_tune, y_tune)

best_k = grid.best_params_['n_neighbors']

print("Optimal k value:", best_k)
print("Highest cross-validation F1-score:", grid.best_score_)


# Cell 9: Visualize F1-Score for Different k Values
k_values = param_grid['n_neighbors']
cv_scores = grid.cv_results_['mean_test_score']

plt.figure(figsize=(7, 4))
plt.scatter(k_values, cv_scores)
plt.plot(k_values, cv_scores)
plt.xlabel("Number of Neighbors (k)")
plt.ylabel("Cross-Validation F1-Score")
plt.title("KNN Parameter Tuning Results")
plt.grid(True, alpha=0.3)
plt.show()


# Cell 10: Build and Test the Optimized KNN Model
knn_best = KNeighborsClassifier(
    n_neighbors=best_k,
    n_jobs=-1
)

knn_best.fit(X_train_sm, y_train_sm)
y_pred_best = knn_best.predict(X_test_scaled)

acc_best = accuracy_score(y_test, y_pred_best)
f1_best = f1_score(y_test, y_pred_best)

print("Optimized KNN (k=%d) - Accuracy: %s" % (best_k, acc_best))
print("Optimized KNN (k=%d) - F1-score: %s" % (best_k, f1_best))

# Display prediction results in detail
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_best))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_best,
    target_names=['Normal', 'Fraud']
))


# Cell 11: Compare the Performance of All Models
results = pd.DataFrame({
    'Model': [
        'KNN k=5 (Original Data)',
        'SMOTE + KNN k=5',
        'SMOTE + KNN Best k=%d' % best_k
    ],
    'Accuracy': [acc_base, acc5, acc_best],
    'F1-Score': [f1_base, f1_5, f1_best]
})

print("\nPerformance Summary:")
print(results.round(4).to_string(index=False))

# Compare accuracy and F1-score visually
results.set_index('Model')[['Accuracy', 'F1-Score']].plot(
    kind='bar',
    figsize=(9, 5)
)

plt.title("Comparison of KNN Models")
plt.ylabel("Performance Score")
plt.xticks(rotation=15, ha='right')
plt.ylim(0, 1.05)
plt.legend(loc='best')
plt.tight_layout()
plt.show()

Dataset dimensions: (0, 355)


KeyError: 'Class'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')